# Beca 18 RAG Chatbot
## Resolución Directoral Ejecutiva N.° 033-2026-MINEDU/VMGI-PRONABEC

In [2]:
!pip install chromadb tiktoken langchain-text-splitters google-genai pypdf python-dotenv tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    F

In [ ]:
# STEP 0: Setup
import os
import sys
import time
import random
import tiktoken
import chromadb
import ipywidgets as widgets
from IPython.display import display, clear_output
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google import genai
from google.genai import types
from dotenv import load_dotenv
from tqdm import tqdm

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found in .env"

client = genai.Client(api_key=GEMINI_API_KEY)

print("pypdf:", __import__('pypdf').__version__)
print("tiktoken:", tiktoken.__version__)
print("chromadb:", chromadb.__version__)
print("ipywidgets:", widgets.__version__)
print("✓ GEMINI_API_KEY loaded")

In [ ]:
# STEP 1: PDF Text Extraction
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = " ".join(text.split())
        full_text += f"\n[PAGE {i+1}]\n{text}\n"
    return full_text

pdf_path = "../data/beca18_reglamento.pdf"
full_text = extract_text_from_pdf(pdf_path)

print(f"Total characters: {len(full_text):,}")
print(f"Total words: {len(full_text.split()):,}")
print(f"\nFirst 500 chars:\n{full_text[:500]}")

In [6]:
# STEP 2: Tokenization and Chunking
enc = tiktoken.get_encoding("cl100k_base")
tokens = enc.encode(full_text)
print(f"Total tokens: {len(tokens):,}")
print(f"Embedding limit: 8,192 tokens")
print(f"Chunk size 800 tokens with 100 overlap is appropriate because:")
print(f"  - Each chunk fits well within the 8,192 token embedding limit")
print(f"  - Overlap preserves context between adjacent chunks")
print(f"  - Small enough to be specific, large enough to be meaningful")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " "]
)

chunks = splitter.split_text(full_text)
metadata = [{"document": "beca18_reglamento", "topic": "Beca18", "language": "es"}
            for _ in chunks]

print(f"\nTotal chunks: {len(chunks)}")
print(f"Average chunk length: {sum(len(c) for c in chunks)/len(chunks):.0f} chars")

Total tokens: 102,723
Embedding limit: 8,192 tokens
Chunk size 800 tokens with 100 overlap is appropriate because:
  - Each chunk fits well within the 8,192 token embedding limit
  - Overlap preserves context between adjacent chunks
  - Small enough to be specific, large enough to be meaningful

Total chunks: 750
Average chunk length: 504 chars


In [7]:
# STEP 3: Embeddings
def embed_documents(texts, max_retries=5):
    embeddings = []
    for i, text in enumerate(tqdm(texts, desc="Embedding docs")):
        for attempt in range(max_retries):
            try:
                result = client.models.embed_content(
                    model="gemini-embedding-001",
                    contents=text,
                    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
                )
                embeddings.append(result.embeddings[0].values)
                time.sleep(1.1)
                break
            except Exception as e:
                wait = (2 ** attempt) + random.uniform(0, 1)
                print(f"Retry {attempt+1} for chunk {i}: {e}. Waiting {wait:.1f}s")
                time.sleep(wait)
    return embeddings

def embed_query(text, max_retries=5):
    for attempt in range(max_retries):
        try:
            result = client.models.embed_content(
                model="gemini-embedding-001",
                contents=text,
                config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY")
            )
            return result.embeddings[0].values
        except Exception as e:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f"Retry {attempt+1}: {e}. Waiting {wait:.1f}s")
            time.sleep(wait)
    raise Exception("Max retries exceeded")

print("\u2713 Embedding functions defined with exponential backoff")

✓ Embedding functions defined with exponential backoff


In [8]:
# STEP 4: Vector Database
chroma_client = chromadb.PersistentClient(path="chroma_db_beca18")

collection_name = "beca18_chunks"
existing = [c.name for c in chroma_client.list_collections()]

if collection_name in existing:
    collection = chroma_client.get_collection(collection_name)
    print(f"\u2713 Loaded existing collection with {collection.count()} documents")
else:
    collection = chroma_client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )
    print("Indexing chunks (this may take a few minutes)...")
    embeddings = embed_documents(chunks)
    ids = [f"chunk_{i}" for i in range(len(chunks))]
    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=chunks,
        metadatas=metadata
    )
    print(f"\u2713 Indexed {collection.count()} documents")

Indexing chunks (this may take a few minutes)...


Embedding docs: 100%|██████████| 750/750 [16:13<00:00,  1.30s/it]


✓ Indexed 750 documents


In [9]:
# STEP 5: Semantic Search
def semantic_search(question, k=5):
    query_embedding = embed_query(question)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )
    output = []
    for i in range(len(results["documents"][0])):
        output.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })
    return output

test_question = "\u00bfCu\u00e1les son los requisitos para postular a Beca 18?"
results = semantic_search(test_question, k=3)
print(f"Top 3 results for: '{test_question}'\n")
for i, r in enumerate(results):
    print(f"[{i+1}] Distance: {r['distance']:.4f}")
    print(f"     Text: {r['text'][:200]}...")
    print()

Top 3 results for: '¿Cuáles son los requisitos para postular a Beca 18?'

[1] Distance: 0.1928
     Text: 16 N° REQUISITO DOCUMENTO DE ACREDITACIÓN o FORMA DE ACREDITACIÓN documento oficial de la IES donde se acredite que mantiene dicha vacante. 2 Haber culminado el nivel secundario de la Educación Básica...

[2] Distance: 0.2146
     Text: . Además, de forma excepcional, las notas contenidas en el Certificado Oficial de Estudios escaneado, el Certificado Oficial de Estudios Digital y la Constancia de Logros de Aprendizaje pueden complem...

[3] Distance: 0.2171
     Text: 34 7. Población Objetivo de Beca 18 y Becas Especiales La población objetivo de Beca 18 Ordinaria está conformada por los jóvenes egresados de la educación secundaria con tercio superior 7 y en situac...



In [10]:
# STEP 6: Grounded Generation
SYSTEM_PROMPT = """Eres un asistente experto en el reglamento de Beca 18 (PRONABEC).
Responde \u00daNICAMENTE bas\u00e1ndote en el contexto proporcionado.
Cuando cites informaci\u00f3n, menciona el n\u00famero de p\u00e1gina si est\u00e1 disponible usando [PAGE N].
Si la informaci\u00f3n no est\u00e1 en el contexto, responde exactamente:
\"El documento no contiene informaci\u00f3n sobre este tema.\"
No uses conocimiento previo ni hagas suposiciones."""

def answer_with_context(question, k=5):
    results = semantic_search(question, k=k)
    context = "\n\n".join([
        f"[Fragmento {i+1} | Distancia: {r['distance']:.3f}]\n{r['text']}"
        for i, r in enumerate(results)
    ])
    prompt = f"""Contexto del reglamento Beca 18:
{context}

Pregunta: {question}

Responde bas\u00e1ndote exclusivamente en el contexto anterior."""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT),
        contents=prompt
    )
    return response.text, results

test_questions = [
    "\u00bfCu\u00e1les son los requisitos de elegibilidad para postular a Beca 18?",
    "\u00bfCu\u00e1les son las modalidades de la beca?",
    "\u00bfCu\u00e1l es el monto del est\u00edpendio mensual?",
    "\u00bfCu\u00e1les son las obligaciones del becario?",
    "\u00bfEn qu\u00e9 casos se puede perder la beca?",
    "\u00bfCu\u00e1l es el precio del iPhone 16 Pro Max?"
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {q}")
    print(f"{'='*60}")
    answer, _ = answer_with_context(q)
    print(f"RESPUESTA: {answer}")
    time.sleep(2)


PREGUNTA: ¿Cuáles son los requisitos de elegibilidad para postular a Beca 18?
RESPUESTA: Los requisitos de elegibilidad para postular a Beca 18, según el contexto proporcionado, son los siguientes:

*   Haber culminado el nivel secundario de la Educación Básica Regular (EBR) o Alternativa (EBA) o Especial (EBE). Los estudios deben ser reconocidos por el Ministerio de Educación [PAGE 16].
*   En el caso de la Beca 18 Ordinaria, el postulante debe haber egresado de la Educación Básica Regular (EBR) o Básica Alternativa (EBA), como máximo en los tres (3) años anteriores al año de la publicación de las Bases (entre 2023 y 2025), con excepción de las personas que acrediten discapacidad, así como los preseleccionados que cumplen lo regulado mediante Resolución Directoral Ejecutiva N.° 086 -2025-MINEDU-VMGI- PRONABEC [PAGE 16].
*   Para la Beca 18 Ordinaria, los jóvenes deben ser egresados de la educación secundaria con tercio superior [PAGE 34].
*   Para la Beca 18 Ordinaria, deben estar en

In [11]:
# STEP 7: Interactive Chat Interface
output_area = widgets.Output()
question_input = widgets.Text(
    placeholder="Escribe tu pregunta sobre Beca 18...",
    layout=widgets.Layout(width="100%")
)
ask_button = widgets.Button(
    description="Consultar",
    button_style="primary",
    layout=widgets.Layout(width="150px")
)
clear_button = widgets.Button(
    description="Limpiar",
    button_style="warning",
    layout=widgets.Layout(width="150px")
)
k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description="Fragmentos:",
    style={"description_width": "initial"}
)

def on_ask(b):
    question = question_input.value.strip()
    if not question:
        return
    with output_area:
        clear_output()
        print(f"\U0001f50d Buscando respuesta para: {question}\n")
        try:
            answer, sources = answer_with_context(question, k=k_slider.value)
            print(f"\U0001f4ac Respuesta:\n{answer}\n")
            accordion_children = []
            for i, s in enumerate(sources):
                text_widget = widgets.HTML(
                    f"<b>P\u00e1gina:</b> (ver texto) | <b>Distancia:</b> {s['distance']:.4f}<br><br>{s['text']}"
                )
                accordion_children.append(text_widget)
            accordion = widgets.Accordion(children=accordion_children)
            for i in range(len(sources)):
                accordion.set_title(i, f"Fuente {i+1} \u2014 distancia: {sources[i]['distance']:.4f}")
            accordion.selected_index = None
            print("\n\U0001f4da Fuentes utilizadas:")
            display(accordion)
        except Exception as e:
            print(f"\u274c Error: {e}")

def on_clear(b):
    question_input.value = ""
    with output_area:
        clear_output()

ask_button.on_click(on_ask)
clear_button.on_click(on_clear)

ui = widgets.VBox([
    widgets.HTML("<h2>\U0001f393 Chatbot Normativa Beca 18</h2><p>Resoluci\u00f3n Directoral Ejecutiva N.\u00b0 033-2026-MINEDU/VMGI-PRONABEC</p>"),
    question_input,
    widgets.HBox([ask_button, clear_button, k_slider]),
    output_area
])
display(ui)